In [51]:
import sacrebleu
import os
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path

In [48]:
from comet import download_model, load_from_checkpoint

ModuleNotFoundError: No module named 'comet'

In [52]:
language_name = "kalamang"

In [53]:
test_data_file = f"../data/gemini_finetune_data/{language_name}/test_inference.jsonl"

In [54]:
test_with_results = f"../data/gemini_finetune_data/{language_name}/test.jsonl" 

In [66]:
with open(test_data_file, 'r', encoding='utf-8') as f:
    test_lines = f.readlines()

print(f"Number of test samples: {len(test_lines)}")

with open(test_with_results, 'r', encoding='utf-8') as f:
    test_with_results_lines = f.readlines()

print(f"Number of test samples with results: {len(test_with_results_lines)}")
    

Number of test samples: 500
Number of test samples with results: 500


In [67]:
test_data = []
for line, result_line in zip(test_lines, test_with_results_lines):
    data_point = json.loads(line)
    result_data_point = json.loads(result_line)
    key = data_point["key"]
    source = data_point["request"]['contents'][0]['parts'][0]['text']
    source_results = result_data_point['contents'][0]['parts'][0]['text']
    target = result_data_point["contents"][1]["parts"][0]['text']
    if source==source_results:
        source = source.strip()
        source = source[source.find(":")+1:].strip()

        test_data.append({
            "key": key,
            "source": source,
            "target": target
        })
    else:
        print(f"Mismatch in source for key {key}")

print(f"Number of valid test samples: {len(test_data)}")


Number of valid test samples: 500


In [68]:
results_dir = f"../data/gemini_final_test_generations/{language_name}/"
results_dir = Path(results_dir)
all_results = {}
for filename in os.listdir(results_dir):
    if filename.endswith(".jsonl"):
        file_path = results_dir / filename
        print(f"Loading results from {file_path}")
        with open(file_path, "r", encoding="utf-8") as f:
            all_results[filename] = []
            for line in f:
                all_results[filename].append(json.loads(line))


    


Loading results from ..\data\gemini_final_test_generations\kalamang\prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10-2025-12-24T22-00-52.jsonl
Loading results from ..\data\gemini_final_test_generations\kalamang\prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_15-2025-12-24T22-48-29.jsonl
Loading results from ..\data\gemini_final_test_generations\kalamang\prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_20-2025-12-24T23-35-27.jsonl
Loading results from ..\data\gemini_final_test_generations\kalamang\prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_5-2025-12-25T00-35-04.jsonl
Loading results from ..\data\gemini_final_test_generations\kalamang\prediction-tuned-kalamang_all_batch_prompts_ADV_400_rule_batch_10-2025-12-25T02-46-34.jsonl
Loading results from ..\data\gemini_final_test_generations\kalamang\prediction-tuned-kalamang_all_batch_prompts_ADV_400_rule_batch_15-2025-12-25T03-10-49.jsonl
Loading results from ..\d

In [69]:
# Load test file
test_file = "../data/gemini_final_test.jsonl"

In [70]:
processed_results = {}

for filename, entries in all_results.items():
    print(f"Processing results from {filename}")
    processed_results[filename] = []
    for entry in entries:
        #print(entry)
        key = entry['key']
        #print(entry)
        try:
            source = entry['request']['contents'][0]['parts'][0]['text']
            translation = entry['response']['candidates'][0]['content']['parts'][0]['text']
        except KeyError:
            print(f"KeyError for entry with key {key} in file {filename}")
            continue
        #print(f"Key: {key}")
        #print(f"Source: {source}")
        #print(f"Translation: {translation}")
        processed_results[filename].append({
            "key": key,
            "source": source,
            "translation": translation
        })
        #print()
    
print(len(processed_results))

Processing results from prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10-2025-12-24T22-00-52.jsonl
Processing results from prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_15-2025-12-24T22-48-29.jsonl
Processing results from prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_20-2025-12-24T23-35-27.jsonl
Processing results from prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_5-2025-12-25T00-35-04.jsonl
Processing results from prediction-tuned-kalamang_all_batch_prompts_ADV_400_rule_batch_10-2025-12-25T02-46-34.jsonl
Processing results from prediction-tuned-kalamang_all_batch_prompts_ADV_400_rule_batch_15-2025-12-25T03-10-49.jsonl
Processing results from prediction-tuned-kalamang_all_batch_prompts_ADV_400_rule_batch_20-2025-12-25T03-32-04.jsonl
Processing results from prediction-tuned-kalamang_all_batch_prompts_ADV_400_rule_batch_5-2025-12-25T03-51-19.jsonl
Processing results from prediction-tuned-kalamang_all_batc

In [71]:
# Create a mapping from key to target for easy lookup
key_to_target = {item['key']: item['target'] for item in test_data}
key_to_source = {item['key']: item['source'] for item in test_data}

# Append target to each processed result
for filename, entries in processed_results.items():
    for entry in entries:
        key = entry['key']
        entry['target'] = key_to_target.get(key, "")
        entry['source'] = key_to_source.get(key, "")
        if entry['target'] == "":
            print(f"Warning: No target found for key {key} in file {filename}")



In [72]:
# Save processed results to new JSONL files
output_dir = f"../data/gemini_final_test_generations_processed/{language_name}/"
os.makedirs(output_dir, exist_ok=True)
for filename, entries in processed_results.items():
    output_file = os.path.join(output_dir, filename)
    with open(output_file, "w", encoding="utf-8") as f:
        for entry in entries:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
    print(f"Saved processed results to {output_file}")

Saved processed results to ../data/gemini_final_test_generations_processed/kalamang/prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10-2025-12-24T22-00-52.jsonl
Saved processed results to ../data/gemini_final_test_generations_processed/kalamang/prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_15-2025-12-24T22-48-29.jsonl
Saved processed results to ../data/gemini_final_test_generations_processed/kalamang/prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_20-2025-12-24T23-35-27.jsonl
Saved processed results to ../data/gemini_final_test_generations_processed/kalamang/prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_5-2025-12-25T00-35-04.jsonl
Saved processed results to ../data/gemini_final_test_generations_processed/kalamang/prediction-tuned-kalamang_all_batch_prompts_ADV_400_rule_batch_10-2025-12-25T02-46-34.jsonl
Saved processed results to ../data/gemini_final_test_generations_processed/kalamang/prediction-tuned-kala

In [64]:
metrics = {}

for filename, entries in processed_results.items():
    print(f"Evaluating results from {filename}")
    metrics[filename] = {}

    references = []
    hypotheses = []
    for entry in entries:
        key = entry['key']
        # Find the corresponding target in test_data
        target_entry = next((item for item in test_data if item["key"] == key), None)
        if target_entry:
            target = target_entry['target']
            translation = entry['translation']
            references.append(target)
            hypotheses.append(translation)
        else:
            print(f"Key {key} not found in test data.")
    
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    chrf = sacrebleu.corpus_chrf(hypotheses, [references])
    chrfpp = sacrebleu.corpus_chrf(hypotheses, [references], word_order=2)

    print(f"BLEU score for {filename}: {bleu.score}")
    print(f"ChrF score for {filename}: {chrf.score}")
    print(f"ChrF++ score for {filename}: {chrfpp.score}")
    metrics[filename]['bleu'] = bleu.score
    metrics[filename]['chrf'] = chrf.score
    metrics[filename]['chrfpp'] = chrfpp.score
    print(  "-----------------------------------")

Evaluating results from prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10-2025-12-24T22-00-52.jsonl
BLEU score for prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10-2025-12-24T22-00-52.jsonl: 0.910456531083165
ChrF score for prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10-2025-12-24T22-00-52.jsonl: 13.912033265255856
ChrF++ score for prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10-2025-12-24T22-00-52.jsonl: 13.072056214389812
-----------------------------------
Evaluating results from prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_15-2025-12-24T22-48-29.jsonl
BLEU score for prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_15-2025-12-24T22-48-29.jsonl: 0.21270310019399621
ChrF score for prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_15-2025-12-24T22-48-29.jsonl: 8.090718305823138
ChrF++ score for prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM